# 26. Q-learning, states, and discounted returns

**Level:** Intermediate to advanced

**Prerequisites:** 25: RL vocabulary and exploration.

[Course index](00_START_HERE.ipynb) | [Previous](25_Reinforcement_Learning_and_Bandits.ipynb) | [Next](27_Neurons_Backpropagation_and_Autograd.ipynb)

**Topics:** Q-learning, MDP, State, Transition, Discount factor, Bellman target, Terminal state.

**Matching script:** `examples\26_Q_Learning_and_MDPs.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A Markov decision process describes states, actions, transition probabilities, rewards, and a
discount factor. The Markov assumption says the chosen state carries the information needed
to model the next transition. A partial observation may require history or a different formalism.

Q-learning estimates the return of an action in a state. Its temporal-difference target uses the
immediate reward plus a discounted best next-state value. A terminal transition must not
bootstrap from an imaginary next step. Learning rate, exploration, coverage, and reward design
influence behavior.

Tabular methods are practical only for small state/action spaces. Deep RL approximates values
or policies with neural networks and introduces additional stability and data-efficiency issues.
Offline RL learns from logged interactions and must handle distribution mismatch; it does not
grant permission to try arbitrary actions on real infrastructure.

## Mathematics or data contract

Q(s,a) <- Q(s,a)+alpha*[r+gamma*max_a_prime Q(s_prime,a_prime)-Q(s,a)], with next value zero at terminal.

## Important limitation

A deterministic four-position line is a teaching MDP, not a realistic network-control simulator.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Learn a route through a tiny safe state space

**Purpose:** Implement a bounded tabular learning loop.

### Read the code in this order
1. Use states 0..3 with 3 terminal and actions left/right.
2. Explore during bounded episodes.
3. Apply a terminal-aware Bellman update.
4. Execute the resulting greedy policy and assert it reaches the goal.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
greedy path: [0, 1, 2, 3]
state-zero Q: [0.8, 0.863]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
Q = np.zeros((4, 2))
for episode in range(350):
    state = 0
    for _ in range(25):
        action = int(rng.integers(2)) if rng.random() < 0.25 else int(np.argmax(Q[state]))
        next_state = max(0, state - 1) if action == 0 else min(3, state + 1)
        terminal = next_state == 3
        reward = 1.0 if terminal else -0.02
        target = reward + (0.0 if terminal else 0.95 * Q[next_state].max())
        Q[state, action] += 0.3 * (target - Q[state, action])
        state = next_state
        if terminal:
            break
state, path = 0, [0]
for _ in range(6):
    action = int(np.argmax(Q[state]))
    state = max(0, state - 1) if action == 0 else min(3, state + 1)
    path.append(state)
    if state == 3:
        break
assert path == [0, 1, 2, 3]
print("greedy path:", path)
print("state-zero Q:", np.round(Q[0], 3).tolist())

greedy path: [0, 1, 2, 3]
state-zero Q: [0.8, 0.863]


### Interpret example 1

The result should demonstrate this purpose: **Implement a bounded tabular learning loop.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a per-step penalty and compare the greedy path; test that terminal states do not bootstrap.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Q-learning` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Q(s,a) <- Q(s,a)+alpha*[r+gamma*max_a_prime Q(s_prime,a_prime)-Q(s,a)], with next value zero at terminal.
- Name the boundary of the demonstration: A deterministic four-position line is a teaching MDP, not a realistic network-control simulator.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `http://incompleteideas.net/book/the-book-2nd.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.